In [1]:
!pip install duckdb
import os
import duckdb
import pandas as pd
import pyarrow.parquet as pq
import logging
from datetime import datetime
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(message)s")
logger = logging.getLogger()

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

In [2]:
read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\pattern_maker_output\top_pps.parquet").head(60)

,country_a_id,country_b_id,length,start_year_a,start_year_b,average_correlation,scdi_count,pps,pps_normalized
0,21,74,62,1960,1960,0.971913,5,22.480731,100.000000
1,21,74,62,1960,1960,0.971913,5,22.480731,100.000000
2,21,74,62,1961,1961,0.969882,5,22.480061,99.996806
3,21,74,62,1961,1961,0.969882,5,22.480061,99.996806
4,74,107,62,1960,1960,0.965909,5,22.478750,99.990559
5,74,107,62,1960,1960,0.965909,5,22.478750,99.990559
6,74,107,62,1961,1961,0.965180,5,22.478509,99.989412
7,74,107,62,1961,1961,0.965180,5,22.478509,99.989412
8,14,74,62,1960,1960,0.964982,5,22.478444,99.989101
9,14,74,62,1960,1960,0.964982,5,22.478444,99.989101


In [3]:
import glob
import dask.dataframe as dd


TOP_PPS_PATTERNS_COUNT = 100_000
pattern_maker_output_path = r"/workspace/qgi\qgi-4.0\data\pattern_maker_output"

# def create_top_pps():
#     """Create a Parquet file with top N patterns based on PPS"""

logger.info(f"Creating 'top_pps.parquet' with top {TOP_PPS_PATTERNS_COUNT} patterns based on PPS...")
patterns_files = glob.glob(os.path.join(pattern_maker_output_path, "*.parquet"))
if not patterns_files:
    logger.warning("No pattern files found. Skipping creation of 'top_pps.parquet'...")

# Read all Parquet files into a Dask DataFrame
logger.info("Reading all Parquet files into a Dask DataFrame...")
ddf = dd.read_parquet(patterns_files)

# Ensure ddf is a Dask DataFrame
if not isinstance(ddf, dd.DataFrame):
    logger.error("ddf is not a Dask DataFrame. Cannot proceed.")


2025-04-16 01:04:18,630 - Creating 'top_pps.parquet' with top 100000 patterns based on PPS...
2025-04-16 01:04:18,635 - Reading all Parquet files into a Dask DataFrame...


In [4]:
# Sort by PPS and select top N patterns
logger.info(f"Selecting top {TOP_PPS_PATTERNS_COUNT} patterns based on PPS...")
sorted_ddf = ddf.sort_values('pps', ascending=False)
sorted_ddf

2025-04-16 01:04:18,753 - Selecting top 100000 patterns based on PPS...


,country_a_id,country_b_id,length,start_year_a,start_year_b,average_correlation,scdi_count,pps,pps_normalized
npartitions=509,,,,,,,,,
,uint16,uint16,uint16,uint16,uint16,float64,int64,float64,float64
,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...


In [5]:
top_patterns_ddf = sorted_ddf.head(TOP_PPS_PATTERNS_COUNT)
top_patterns_ddf

,country_a_id,country_b_id,length,start_year_a,start_year_b,average_correlation,scdi_count,pps,pps_normalized
769152,21,74,62,1960,1960,0.971913,5,22.480731,100.000000
0,21,74,62,1960,1960,0.971913,5,22.480731,100.000000
1,21,74,62,1961,1961,0.969882,5,22.480061,99.996806
713025,21,74,62,1961,1961,0.969882,5,22.480061,99.996806
157609,74,107,62,1960,1960,0.965909,5,22.478750,99.990559
...,...,...,...,...,...,...,...,...,...
1401554,18,205,62,1962,1961,0.907583,3,21.779502,96.658544
947014,78,157,62,1962,1960,0.907582,3,21.779502,96.658543
1353772,18,147,62,1962,1961,0.907582,3,21.779502,96.658542
42664,102,158,61,1962,1962,0.877279,4,21.779502,96.658542


In [6]:
output_path = os.path.join(pattern_maker_output_path, "top_pps.parquet")
logger.info(f"Saving 'top_pps.parquet' to {output_path}...")
top_patterns_ddf.to_parquet(
    output_path,
    index=False,
    compression="brotli"
)
logger.info(f"Saved 'top_pps.parquet' to {output_path}")

2025-04-16 01:08:33,182 - Saving 'top_pps.parquet' to /workspace/qgi\qgi-4.0\data\pattern_maker_output\top_pps.parquet...
2025-04-16 01:08:33,508 - Saved 'top_pps.parquet' to /workspace/qgi\qgi-4.0\data\pattern_maker_output\top_pps.parquet
